In [61]:
import torchvision
print(torchvision.__file__)


C:\Users\yathi\anaconda3\envs\pytorch\lib\site-packages\torchvision\__init__.py


In [62]:
import sys
print(sys.executable)

C:\Users\yathi\anaconda3\envs\pytorch\python.exe


In [63]:
from torchvision.ops import nms
print(nms)


<function nms at 0x0000019238F3AEF0>


In [64]:
import torchvision
print(torchvision.__file__)
print(torchvision.__version__)


C:\Users\yathi\anaconda3\envs\pytorch\lib\site-packages\torchvision\__init__.py
0.26.0+cpu


In [65]:
import torch

import numpy as np
from torchvision import datasets
import torchvision.transforms as transforms
from torch.utils.data import Subset

In [66]:
# number of subprocesses to use for data loading
num_workers = 0
# how many samples per batch to load
batch_size = 32

# convert data to torch.FloatTensor
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.1307,), (0.3081,))]
)

# choose the training and test datasets
train_data = datasets.MNIST(
    root='data',
    train=True,
    download=True,
    transform=transform
)
test_data = datasets.MNIST(
    root='data', train=False,
    download=True, transform=transform
)

100%|███████████████████████████████████████████████████████████████████████████████████████████| 9.91M/9.91M [00:14<00:00, 677kB/s]
100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 28.9k/28.9k [00:00<?, ?B/s]
100%|███████████████████████████████████████████████████████████████████████████████████████████| 1.65M/1.65M [00:02<00:00, 749kB/s]
100%|██████████████████████████████████████████████████████████████████████████████████████████| 4.54k/4.54k [00:00<00:00, 1.56MB/s]


In [67]:
# number of teachers to essemble
num_teachers = 100

def get_data_loaders(train_data, num_teachers=10):
    """Simple partitioning algorithm that returns the right portion of the data
    needed by a given teacher out of a certain number of teachers.

    Each teacher model will get a disjoint subset of the training data.
    """
    teacher_loaders = []
    data_size = len(train_data) // num_teachers

    for i in range(num_teachers):
        indices = list(range(i * data_size, (i+1) * data_size))
        subset_data = Subset(train_data, indices)
        loader = torch.utils.data.DataLoader(
            subset_data,
            batch_size=batch_size,
            num_workers=num_workers
        )
        teacher_loaders.append(loader)

    return teacher_loaders

teacher_loaders = get_data_loaders(train_data, num_teachers)

In [68]:
student_train_data = Subset(test_data, list(range(9000)))
student_test_data = Subset(test_data, list(range(9000, 10000)))

student_train_loader = torch.utils.data.DataLoader(
    student_train_data, batch_size=batch_size, 
    num_workers=num_workers
)
student_test_loader = torch.utils.data.DataLoader(
    student_test_data, batch_size=batch_size, 
    num_workers=num_workers
)

In [69]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 10, kernel_size=5)
        self.conv2 = nn.Conv2d(10, 20, kernel_size=5)
        self.conv2_drop = nn.Dropout2d()
        self.fc1 = nn.Linear(320, 50)
        self.fc2 = nn.Linear(50, 10)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.conv2_drop(self.conv2(x)), 2))
        x = x.view(-1, 320)
        x = F.relu(self.fc1(x))
        x = F.dropout(x, training=self.training)
        x = self.fc2(x)
        #return F.log_softmax(x)
        return F.log_softmax(x, dim=1)

    

In [70]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

def train(model, trainloader, criterion, optimizer, epochs=10, print_every=120):
    model.to(device)
    steps = 0
    running_loss = 0
    for e in range(epochs):
        # Model in training mode, dropout is on
        model.train()
        for images, labels in trainloader:
            images, labels = images.to(device), labels.to(device)
            steps += 1
            
            optimizer.zero_grad()
            
            output = model.forward(images)
            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()


In [71]:
def predict(model, dataloader):
    outputs = torch.zeros(0, dtype=torch.long).to(device)
    model.to(device)
    model.eval()
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        output = model.forward(images)
        ps = torch.argmax(torch.exp(output), dim=1)
        outputs = torch.cat((outputs, ps))
    
    return outputs

In [72]:
from tqdm.notebook import trange

# Instantiate and train the models for each teacher
def train_models(num_teachers):
    models = []
    for t in trange(num_teachers):
        model = Net()
        criterion = nn.NLLLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.003)
        train(model, teacher_loaders[t], criterion, optimizer)
        models.append(model)
    return models

models = train_models(num_teachers) 

  0%|          | 0/100 [00:00<?, ?it/s]

In [73]:
import numpy as np

In [74]:
# define standard deviation for noise
standard_deviation = 5.0

In [75]:
def aggregated_teacher(models, data_loader, standard_deviation=1.0):
    preds = torch.torch.zeros((len(models), 9000), dtype=torch.long)
    print('Running teacher predictions...')
    for i, model in enumerate(models):
        results = predict(model, data_loader)
        preds[i] = results
    
    print('Calculating aggregates...')
    labels = np.zeros(preds.shape[1]).astype(int)
    for i, image_preds in enumerate(np.transpose(preds)):
        label_counts = np.bincount(image_preds, minlength=10).astype(float)
        label_counts += np.random.normal(0, standard_deviation, len(label_counts))
        labels[i] = np.argmax(label_counts)
    
    return preds.numpy(), np.array(labels)

In [76]:
teacher_models = models
preds, student_labels = aggregated_teacher(teacher_models, student_train_loader, standard_deviation)

Running teacher predictions...
Calculating aggregates...


In [77]:
def student_loader(student_train_loader, labels):
    for i, (data, _) in enumerate(iter(student_train_loader)):
        #yield data, torch.from_numpy(labels[i*len(data):(i+1)*len(data)])
        yield data, torch.from_numpy(labels[i*len(data):(i+1)*len(data)]).long()


In [78]:
student_model = Net()
criterion = nn.NLLLoss()
optimizer = optim.Adam(student_model.parameters(), lr=0.001)
epochs = 10
student_model.to(device)
steps = 0
running_loss = 0
for e in range(epochs):
    student_model.train()
    train_loader = student_loader(student_train_loader, student_labels)
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        steps += 1

        optimizer.zero_grad()
        output = student_model.forward(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if steps % 50 == 0:
            test_loss = 0
            accuracy = 0
            student_model.eval()
            with torch.no_grad():
                for images, labels in student_test_loader:
                    #images, labels = images.to(device), labels.to(device)
                    images = images.to(device)
                    labels = labels.long().to(device)     # ← FIX

                    log_ps = student_model(images)
                    test_loss += criterion(log_ps, labels).item()
                    
                    ps = torch.exp(log_ps)
                    top_p, top_class = ps.topk(1, dim=1)
                    equals = top_class == labels.view(*top_class.shape)
                    accuracy += torch.mean(equals.type(torch.FloatTensor))
            student_model.train()
            print("Epoch: {}/{}.. ".format(e+1, epochs),
                  "Training Loss: {:.3f}.. ".format(running_loss/len(student_train_loader)),
                  "Test Loss: {:.3f}.. ".format(test_loss/len(student_test_loader)),
                  "Test Accuracy: {:.3f}".format(accuracy/len(student_test_loader)))
            running_loss = 0

Epoch: 1/10..  Training Loss: 0.371..  Test Loss: 1.433..  Test Accuracy: 0.688
Epoch: 1/10..  Training Loss: 0.214..  Test Loss: 0.638..  Test Accuracy: 0.821
Epoch: 1/10..  Training Loss: 0.138..  Test Loss: 0.461..  Test Accuracy: 0.844
Epoch: 1/10..  Training Loss: 0.095..  Test Loss: 0.396..  Test Accuracy: 0.888
Epoch: 1/10..  Training Loss: 0.093..  Test Loss: 0.302..  Test Accuracy: 0.914
Epoch: 2/10..  Training Loss: 0.116..  Test Loss: 0.365..  Test Accuracy: 0.923
Epoch: 2/10..  Training Loss: 0.093..  Test Loss: 0.275..  Test Accuracy: 0.927
Epoch: 2/10..  Training Loss: 0.080..  Test Loss: 0.253..  Test Accuracy: 0.932
Epoch: 2/10..  Training Loss: 0.077..  Test Loss: 0.257..  Test Accuracy: 0.938
Epoch: 2/10..  Training Loss: 0.058..  Test Loss: 0.243..  Test Accuracy: 0.930
Epoch: 2/10..  Training Loss: 0.051..  Test Loss: 0.235..  Test Accuracy: 0.934
Epoch: 3/10..  Training Loss: 0.105..  Test Loss: 0.240..  Test Accuracy: 0.931
Epoch: 3/10..  Training Loss: 0.078..  T

In [79]:
preds.shape

(100, 9000)

In [81]:
# put together the counts matrix:
clean_votes = []
for image_preds in np.transpose(preds):
    label_counts = np.bincount(image_preds, minlength=10).astype(float)
    clean_votes.append(label_counts)

#clean_votes = np.array(label_counts)
clean_votes = np.array(clean_votes)


In [82]:
print(clean_votes.shape)

(9000, 10)


In [83]:
with open('clean_votes.npy', 'wb') as file_obj:
  np.save(file_obj, clean_votes)

In [84]:
standard_deviation

5.0

In [85]:
import os
os.getcwd()


'C:\\ZDrive\\Yathish\\Potomac\\AIT 620\\LabWorks\\PythonProject\\chapter11\\privacy-master\\research\\pate_2018'

In [88]:
%cd "C:/ZDrive/Yathish/Potomac/AIT 620/LabWorks/PythonProject/chapter11/privacy-master/research/pate_2018/"

C:\ZDrive\Yathish\Potomac\AIT 620\LabWorks\PythonProject\chapter11\privacy-master\research\pate_2018


In [89]:
!python -m ICLR2018.smooth_sensitivity_table --sigma2=5.0 --counts_file="C:/ZDrive/Yathish/Potomac/AIT 620/LabWorks/PythonProject/chapter11/clean_votes.npy" --delta=1e-5

Reading raw votes from C:/ZDrive/Yathish/Potomac/AIT 620/LabWorks/PythonProject/chapter11/clean_votes.npy


Traceback (most recent call last):
  File "C:\Users\yathi\anaconda3\envs\pytorch\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\yathi\anaconda3\envs\pytorch\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "C:\ZDrive\Yathish\Potomac\AIT 620\LabWorks\PythonProject\chapter11\privacy-master\research\pate_2018\ICLR2018\smooth_sensitivity_table.py", line 358, in <module>
    app.run(main)
  File "C:\Users\yathi\anaconda3\envs\pytorch\lib\site-packages\absl\app.py", line 367, in run
    _run_main(main, args)
  File "C:\Users\yathi\anaconda3\envs\pytorch\lib\site-packages\absl\app.py", line 312, in _run_main
    sys.exit(main(argv))
  File "C:\ZDrive\Yathish\Potomac\AIT 620\LabWorks\PythonProject\chapter11\privacy-master\research\pate_2018\ICLR2018\smooth_sensitivity_table.py", line 323, in main
    votes, baseline = _load_votes(FLAGS.counts_file, FLAGS.baseline_file,
  File "C:\ZDrive\Yathish\Potomac\AIT 

In [90]:
!python -m ICLR2018.smooth_sensitivity_table --sigma2=5.0 --counts_file="C:/ZDrive/Yathish/Potomac/AIT 620/LabWorks/PythonProject/chapter11/privacy-master/research/pate_2018/clean_votes.npy" --delta=1e-5


Reading raw votes from C:/ZDrive/Yathish/Potomac/AIT 620/LabWorks/PythonProject/chapter11/privacy-master/research/pate_2018/clean_votes.npy
Shape of the votes matrix = (9000, 10)
Process all 9000 input rows. (Use --queries flag to truncate.)
queries = 1000, E[answered] = 1000.00, E[eps] = 8.487 (std = 0.79429) at order = 4.00 (contribution from delta = 3.838)
queries = 2000, E[answered] = 2000.00, E[eps] = 14.796 (std = 0.96483) at order = 3.00 (contribution from delta = 5.756)
queries = 3000, E[answered] = 3000.00, E[eps] = 18.764 (std = 0.97701) at order = 2.50 (contribution from delta = 7.675)
queries = 4000, E[answered] = 4000.00, E[eps] = 22.387 (std = 1.12695) at order = 2.50 (contribution from delta = 7.675)
queries = 5000, E[answered] = 5000.00, E[eps] = 27.218 (std = 1.30250) at order = 2.50 (contribution from delta = 7.675)
queries = 6000, E[answered] = 6000.00, E[eps] = 29.457 (std = 1.12116) at order = 2.00 (contribution from delta = 11.513)
queries = 7000, E[answered] = 70